# ComplaintIQ - supervised model at full scale (`07a_mllib_supervised`)

`04`-`06` trained sklearn on a **sample** pulled to the driver, capped by one machine's memory. This
notebook trains a **distributed Spark MLlib** logistic regression on the **entire ~16.5M rows** with
`06`'s engineered features - no sampling, no `.toPandas()`.

Companion notebook: **`07b_mllib_unsupervised.ipynb`** (full-corpus clustering). They are split so
each model trains in its own session and stays under the serverless Spark-Connect **1GB model-cache
cap** - a real Free Edition limit that a single combined notebook overflows.

- **Target:** `monetary_relief`, ranked for a review queue. **Metric:** lift at top-1% / 5% / 10% + PR-AUC.

## How to read this notebook
Everything stays in Spark; only scalar metrics reach the driver. Same leakage rules as `02`
section 13 - intake-only fields, chronological split. Encodings are learned on the **training
window only** and joined to both splits (the leakage-safe, scalable form of `06`'s out-of-fold
target encoding).

> **Note (why encode, not one-hot):** high-cardinality fields like `company` / `zip_code` one-hot
> into thousands of columns, which bloats the fitted model past the 1GB serverless cache. Encoding
> each categorical as **two dense numbers** (its train relief rate and its frequency) keeps the
> model tiny while preserving the signal - the same trick `06` used for lift.

> **Go deeper:**
> - [Spark ML Pipelines](https://spark.apache.org/docs/latest/ml-pipeline.html): *Transformers/Estimators chained into one fitted model, ~12 min.*
> - [Target (mean) encoding](https://contrib.scikit-learn.org/category_encoders/targetencoder.html): *replacing a category with its target rate, and the leakage guard, ~8 min.*

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame as SparkDataFrame
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, NGram, HashingTF, IDF, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

RANDOM_STATE = 42
print("Spark", spark.version)

---
## 1. Load only the columns this model needs

Reading a narrow projection keeps the DataFrame (and everything Spark Connect ships) small. We need
the target, the narrative, the intake categoricals, and the date for the split - nothing else.

In [ ]:
from pathlib import Path

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
path = data_dir / "complaints.parquet"

CAT = [
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "submitted_via",
    "state",
    "tags",
    "company",
    "zip_code",
]
NEEDED = ["monetary_relief", "complaint_text", *CAT]  # intake-only; no leakage fields
df = (
    spark.read.parquet(str(path))
    .select(*NEEDED, F.to_date("date_received").alias("date_received"))
    .dropna(subset=["date_received"])
    .fillna({"complaint_text": "", **{c: "MISSING" for c in CAT}})
)

# No .cache(): serverless disallows PERSIST and manages caching itself.
df = df.withColumn("epoch", F.datediff("date_received", F.lit("1970-01-01")))
cut = df.approxQuantile("epoch", [0.80], 0.001)[0]
train = df.filter(F.col("epoch") <= cut).drop("epoch")
test = df.filter(F.col("epoch") > cut).drop("epoch")
label_col = "monetary_relief"
print(f"train rows: {train.count():,}  |  test rows: {test.count():,}")

---
## 2. Engineered features (leakage-safe, dense)

`06`'s features rebuilt as Spark ops that keep the model small:
- **Target + frequency encoding** of **every** categorical (train-only group means/counts joined to
  both splits). Two dense columns per field - no one-hot, so `company`/`zip_code` cost 2 numbers, not
  thousands.
- **Interaction encodings** - `product x submitted_via`, `product x issue`, also target-encoded.
- **Text-shape** - length, caps ratio, `XX`-redaction density, has-dollar (native functions).

> **Note:** `06`'s **character n-grams** have no native Spark transformer (they need a slow UDF at
> 16.5M rows), so they are omitted here; word unigrams+bigrams (section 3) are the native substitute.

In [ ]:
# Interaction columns first, so they can be encoded like any categorical.
def add_interactions(frame: SparkDataFrame) -> SparkDataFrame:
    return frame.withColumn(
        "product_x_submitted_via", F.concat_ws("|", "product", "submitted_via")
    ).withColumn("product_x_issue", F.concat_ws("|", "product", "issue"))


train = add_interactions(train)
test = add_interactions(test)

ENC = CAT + ["product_x_submitted_via", "product_x_issue"]
prior = train.agg(F.mean(label_col)).first()[0]
n_train = train.count()

# Train-only encoding tables (relief rate + frequency), joined to both splits -> no leakage.
enc_tables = {}
for col in ENC:
    enc_tables[col] = train.groupBy(col).agg(
        F.mean(label_col).alias(col + "_te"),
        (F.count(F.lit(1)) / F.lit(n_train)).alias(col + "_fe"),
    )


def encode(frame: SparkDataFrame) -> SparkDataFrame:
    out = frame
    for col, table in enc_tables.items():
        out = out.join(table, on=col, how="left")
    fills = {}
    for col in ENC:
        fills[col + "_te"] = prior
        fills[col + "_fe"] = 0.0
    out = out.fillna(fills)
    # text-shape (native, no UDF)
    return (
        out.withColumn("len_log", F.log1p(F.length("complaint_text")))
        .withColumn(
            "caps_ratio",
            F.length(F.regexp_replace("complaint_text", "[^A-Z]", ""))
            / F.greatest(F.length("complaint_text"), F.lit(1)),
        )
        .withColumn("redaction", F.log1p(F.expr("regexp_count(complaint_text, 'XX')")))
        .withColumn("has_dollar", F.col("complaint_text").contains("$").cast("double"))
    )


train_fe = encode(train)
test_fe = encode(test)
DENSE = (
    [c + "_te" for c in ENC]
    + [c + "_fe" for c in ENC]
    + ["len_log", "caps_ratio", "redaction", "has_dollar"]
)
print(f"dense engineered features: {len(DENSE)} columns (no one-hot)")

---
## 3. Train the model (full data)

A `Pipeline`: text (`Tokenizer` -> unigrams + `NGram` bigrams -> modest `HashingTF` -> `IDF`) plus
the dense engineered columns, assembled and fed to a class-weighted `LogisticRegression`. The only
large block is the hashed text; the metadata is all dense, so the fitted model stays well under 1GB.

> **Go deeper:**
> - [Spark ML classification](https://spark.apache.org/docs/latest/ml-classification-regression.html): *LogisticRegression regParam / weightCol / class weights, ~12 min.*
> - [HashingTF (feature hashing)](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.HashingTF.html): *hashing text to a fixed size instead of an exact vocabulary, ~6 min.*

In [ ]:
train_fe = train_fe.withColumn(
    "sample_weight", F.when(F.col(label_col) == 1, 1.0 / prior).otherwise(1.0 / (1 - prior))
)

tokenizer = Tokenizer(inputCol="complaint_text", outputCol="unigrams")
bigram = NGram(n=2, inputCol="unigrams", outputCol="bigrams")
hashing_tf_uni = HashingTF(inputCol="unigrams", outputCol="tf_unigram", numFeatures=2**15)
hashing_tf_bi = HashingTF(inputCol="bigrams", outputCol="tf_bigram", numFeatures=2**15)
idf_unigram = IDF(inputCol="tf_unigram", outputCol="tfidf_unigram")
idf_bigram = IDF(inputCol="tf_bigram", outputCol="tfidf_bigram")
assembler = VectorAssembler(
    inputCols=["tfidf_unigram", "tfidf_bigram"] + DENSE, outputCol="features"
)
logreg = LogisticRegression(
    featuresCol="features", labelCol=label_col, weightCol="sample_weight", maxIter=50, regParam=1e-3
)

model = Pipeline(
    stages=[
        tokenizer,
        bigram,
        hashing_tf_uni,
        hashing_tf_bi,
        idf_unigram,
        idf_bigram,
        assembler,
        logreg,
    ]
).fit(train_fe)

---
## 4. Evaluate vs the baseline bar

PR-AUC / ROC-AUC from MLlib's evaluator; **lift at top-1% / 5% / 10%** via `approxQuantile` + filter (no global
sort, so it scales). Same bar as `04`-`06`: beat the product-bucket heuristic (top-10% lift is capped at 10x, so smaller queues like top-1% show more).

In [ ]:
pred = model.transform(test_fe).withColumn("p", vector_to_array("probability")[1])

pr = BinaryClassificationEvaluator(
    labelCol=label_col, rawPredictionCol="probability", metricName="areaUnderPR"
).evaluate(model.transform(test_fe))
roc = BinaryClassificationEvaluator(
    labelCol=label_col, rawPredictionCol="probability", metricName="areaUnderROC"
).evaluate(model.transform(test_fe))
base = pred.agg(F.mean(label_col)).first()[0]
# Lift at several queue sizes (one approxQuantile call for all cutoffs). top-10% lift is capped
# at 10x by construction; smaller queues have higher ceilings and show the ranking power the 10%
# slice cannot express. Threshold + filter -> distributed, no global sort.
quantiles = pred.approxQuantile("p", [0.99, 0.95, 0.90], 0.001)  # top-1% / 5% / 10% score cutoffs
lifts = {}
for k, thr in zip((0.01, 0.05, 0.10), quantiles):
    prec_k = pred.filter(F.col("p") >= thr).agg(F.mean(label_col)).first()[0]
    lifts[k] = (prec_k, (prec_k / base if base else float("nan")))
prec, lift = lifts[0.10]  # keep top-10% as the headline precision/lift
print(f"full-data MLlib logistic regression (engineered features)")
print(f"  test base rate: {base:.4%}")
print(f"  PR-AUC={pr:.4f}  ROC-AUC={roc:.4f}")
print(f"  lift@1%={lifts[0.01][1]:.1f}x  @5%={lifts[0.05][1]:.1f}x  @10%={lifts[0.10][1]:.1f}x")

# Persist metrics to the volume so results are retrievable outside the run (notebook stdout
# is not exposed via the jobs API).
import json as _json, time

metrics = {
    "notebook": "07a_mllib_supervised",
    "rows_train": int(train.count()),
    "rows_test": int(test.count()),
    "base_rate": float(base),
    "pr_auc": float(pr),
    "roc_auc": float(roc),
    "top10_precision": float(prec),
    "lift_1pct": float(lifts[0.01][1]),
    "lift_5pct": float(lifts[0.05][1]),
    "lift_10pct": float(lifts[0.10][1]),
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
out = "/Volumes/workspace/complaintiq/data/metrics_07a.json"
dbutils.fs.put(out, _json.dumps(metrics, indent=2), overwrite=True)
print("wrote", out)

> **What you're seeing:** the fused text + engineered-metadata model `06` pointed to, trained on all
> ~13M training rows and scored on the held-out recent months.
>
> **Why it matters:** these are the real full-data numbers `06`'s sample was estimating, on the same
> top-1% / 5% / 10% lift / PR-AUC bar - the operating point a deployed queue would actually use.

---
## 5. Takeaways

> - **Scale:** trains on all ~16.5M rows, no `.toPandas()` - the driver-memory ceiling of `04`-`06`
>   is gone.
> - **Small model:** every categorical is dense target/frequency-encoded (not one-hot), so the
>   fitted model fits the serverless 1GB Spark-Connect ML-cache cap.
> - **Features:** `06`'s target/frequency encoding, interactions, text-shape, and word unigrams+
>   bigrams, learned on the train window and joined leakage-safe.
> - **Gaps:** hashed text features (collisions) not an exact vocabulary; char n-grams omitted (no
>   native transformer).

Full-corpus clustering is in **`07b_mllib_unsupervised.ipynb`**.